In [ ]:
from scripts.sweep import run_task
server = "mine"

In [ ]:
# train cl diff
!{bash} scripts/run.sh --script train.train_single --server {server} --method cl

In [ ]:
# train fed diff os
from scripts.sweep import run_sweep
script = "train.train_single"
max_jobs = 2
grid = {
    "method": [f"client{i}_os" for i in range(5)]
}
run_sweep(grid, server, script, max_jobs)

In [ ]:
# train cl diff by syn data
!{bash} scripts/run.sh --script train.train_single --server {server} --method cl_distill

In [ ]:
# syn cl
!{bash} scripts/run.sh --script synthesis.syn_cl --server {server}

In [ ]:
# syn ws
run_task("synthesis.syn_ws", server)

In [ ]:
# syn dfd
run_task("synthesis.syn_dfd", server)


Generating:   0%|          | 0/20 [00:00<?, ?it/s]F:\anaconda\envs\ttt\lib\site-packages\diffusers\models\attention_processor.py:1279: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  hidden_states = F.scaled_dot_product_attention(


In [ ]:
# compute fid

import logging

from cleanfid import fid

client_wise = True
client_wise = False


def compute_fid(img_path):
    score = fid.compute_fid(
        img_path,
        dataset_name="cifar10",
        dataset_res=32,
        dataset_split="train",
        num_workers=0,
    )

    print(f"FID: {score:.4f}")

client_wise = False
img_path = "results/cifar10/imgs/avg_sample"
if client_wise:
    for client in range(5):
        logging.info(f"Client {client}")
        compute_fid(f'{img_path}/{client}')
else:
    compute_fid(img_path)